# ***First AI Agents***

In [4]:
import os
from groq import Groq
from dotenv import load_dotenv
import json 

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables")



# ***Building my own tool***

In [5]:
# This fucntion will be used as a tool
def get_temperature_based_on_city(city: str) -> str :
    """ Get the temperature based on the city name. """
    # Ideally this should be an API call to a weather service.
    print("Tool function is getting called")
    if not city:
        return "City name is required."
    if city.lower() == "france":
        return "20°C"
    if city.lower() == "new york":
        return "25°C"
    if city.lower() == "london":
        return "15°C"
    return "City not found."

# ***Tool schema for the weather tool***

In [6]:
# Tool schema for the weather tool
weather_tool_schema = {
    "name": "get_temperature_based_on_city",
    "description": "Get the temperature based on the city name.",
    "type": "function",
    "function": {
        "name": "get_temperature_based_on_city",
        "description": "Get the temperature based on the city name.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city to get the temperature for."
                }
            },
            "required": ["city"]
        }
    }
}

# ***Creating our own agent***

In [7]:
class JarvisAgent:
    def __init__(self, client: Groq, model: str, system: str = "", tools: list | None = None) -> None:
        self.client = client
        self.model = model
        self.messages: list = []
        self.tools = tools if tools is not None else []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message: str= ""):
        if message:
            self.messages.append({"role": "user", "content": message})
        final_assistant_content = self.execute()
        if final_assistant_content:
            self.messages.append({"role": "assistant", "content": final_assistant_content})
        return final_assistant_content

    def execute(self):
        while True:
            completion = self.client.chat.completions.create(
                model = self.model,
                messages = self.messages,
                tools = self.tools,
                tool_choice = "auto" #Let the model decide when to use tools
            )

            response_message = completion.choices[0].message

            if response_message.tool_calls:
                self.messages.append(response_message)

                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    tool_output_content = f"Tool '{function_name}' not found."
                    if function_name in globals() and callable(globals()[function_name]):
                        function_to_call = globals()[function_name]
                        executed_output = function_to_call(**function_args)
                        tool_output_content = str(executed_output)

                    tool_outputs.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": tool_output_content,
                        }
                    )

                self.messages.extend(tool_outputs)
                continue

            return response_message.content

# ***Tool Call through own Agent***

In [8]:
client = Groq(api_key=api_key)
query = "What is the weather in New York right now?"


personal_agent = JarvisAgent(
    client = client,
    model = "openai/gpt-oss-120b",
    system = "You are a helpful assistant named Alex",
    tools = [weather_tool_schema]
)

response = personal_agent(query)
print("Messages", personal_agent.messages)
print("Response from Alex: ", response)

Tool function is getting called
Messages [{'role': 'system', 'content': 'You are a helpful assistant named Alex'}, {'role': 'user', 'content': 'What is the weather in New York right now?'}, ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks: "What is the weather in New York right now?" We have a function to get temperature based on city. Probably we can call get_temperature_based_on_city with city "New York". The function likely returns temperature (maybe also other info). We\'ll call it.', tool_calls=[ChatCompletionMessageToolCall(id='fc_5801c930-95e1-417f-8dc4-16c1c51870c6', function=Function(arguments='{"city":"New York"}', name='get_temperature_based_on_city'), type='function')]), {'tool_call_id': 'fc_5801c930-95e1-417f-8dc4-16c1c51870c6', 'role': 'tool', 'name': 'get_temperature_based_on_city', 'content': '25°C'}, {'role': 'assistant', 'content': 'The current temperature in New\u202fYork is abou